<center><h1> PR Streak </h1></center>

The following code was written by [Thomas Stokes](https://github.com/ThomasStokes1998), WCAID [2017STOK03](https://www.worldcubeassociation.org/persons/2017STOK03). Full credit goes to him.

It calculates the longest streak of consecutive WCA competitions with a <b>personal record</b> (PR) for each competitor.

### Imports & functions

In [1]:
import pandas as pd
from datetime import date
import numpy as np

low_memory = False
results = pd.read_csv('WCA_export_Results.tsv', delimiter='\t')
competitions = pd.read_csv('WCA_export_Competitions.tsv', delimiter='\t')


In [2]:
def mbfPoints(p: int, oldstyle: bool=False) -> float:
    if oldstyle:
        tt = p % 100_000
        ss = 199 - p // 10_000_000
        aa = (p // 100_000) % 100
        if tt == 99_999:
            tt = min(600 * aa, 3_600)
        return ss - aa + 1 - tt / 3_600
    else:
        dd = 99 - p // 10_000_000
        mm = p % 100
        tt = (p // 100) % 100_000
        aa = dd + 2 * mm
        if tt == 99_999:
            tt = min(600 * aa, 3_600)
        if aa < 6:
            return dd + 1 - tt / (600 * aa)
        else:
            return dd + 1 - tt / 3_600


### Dictionaries to keep track of all the data

In [3]:
compdates = {"ongoing":date.today()}
for i, comp in enumerate(competitions.id):
    compdates[comp] = date(competitions.year[i], competitions.month[i], competitions.day[i])


In [4]:
personnames = {}
maxstreaks = {}
maxstart = {}
maxend = {}
currentcomp = {}
startcomps = {}
currentstreak = {}
gotpr = {}
personprs = {}

### Code

In [5]:
def updatedicts(event: str, wcaid: str, single: int, average: int):
    # Add a new event 
    if event not in personprs[wcaid]:
        if event == "333mbf":
            personprs[wcaid][event] = 0
        elif event == "333mbo":
            personprs[wcaid][event] = 0
        else:
            personprs[wcaid][event] = [360_000, 360_000]
    # Check if a new PR has been set
    if event == "333mbf":
        if single > 0 and personprs[wcaid][event] <= mbfPoints(single):
            personprs[wcaid][event] = mbfPoints(single)
            gotpr[wcaid] = True
    elif event == "333mbo":
        if single > 0 and personprs[wcaid][event] <= mbfPoints(single, True):
            personprs[wcaid][event] = mbfPoints(single, True)
            gotpr[wcaid] = True
    else:
        if single > 0 and personprs[wcaid][event][0] >= single:
            personprs[wcaid][event][0] = single
            gotpr[wcaid] = True
        if average > 0 and personprs[wcaid][event][1] >= average:
            personprs[wcaid][event][1] = average
            gotpr[wcaid] = True

Change the country in the first row below to the country of choice or to "world" for global PR straks

In [6]:
def main(country: str = "Italy"):
    if country == "World":
        res = results
    else:
        res = results[results.personCountryId == country].reset_index(drop="index")
    # Order the res dataframe by date
    res["date"] = res["competitionId"].apply(lambda x: compdates[x])
    res = res.sort_values("date").reset_index(drop="index")
    for i, wcaid in enumerate(res.personId):
        comp = res.competitionId[i]
        event = res.eventId[i]
        single = res.best[i]
        average = res.average[i]
        if wcaid not in personnames:
            # Initial values for a new id
            personnames[wcaid] = res.personName[i]
            maxstreaks[wcaid] = 0
            maxstart[wcaid] = "none"
            maxend[wcaid] = "none"
            startcomps[wcaid] = "none"
            currentcomp[wcaid] = comp
            currentstreak[wcaid] = 0
            gotpr[wcaid] = False
            personprs[wcaid] = {}
        if currentcomp[wcaid] == comp:
            updatedicts(event, wcaid, single, average)
        else:
            if gotpr[wcaid]:
                if currentstreak[wcaid] == 0:
                    startcomps[wcaid] = currentcomp[wcaid]
                currentstreak[wcaid] += 1
            else:
                c = currentstreak[wcaid]
                if c > maxstreaks[wcaid]:
                    maxstreaks[wcaid] = c
                    maxstart[wcaid] = startcomps[wcaid]
                    maxend[wcaid] = currentcomp[wcaid]
                currentstreak[wcaid] = 0
            # Reset values for a new competition
            gotpr[wcaid] = False
            currentcomp[wcaid] = comp
            updatedicts(event, wcaid, single, average)
    # Update ongoing streaks
    for wcaid in gotpr:
        if maxstart[wcaid] == "none":
            maxstart[wcaid] = currentcomp[wcaid]
            maxend[wcaid] = currentcomp[wcaid]
        if gotpr[wcaid]:
            if currentstreak[wcaid] == 0:
                startcomps[wcaid] = currentcomp[wcaid]
            currentstreak[wcaid] += 1
            c = currentstreak[wcaid]
            if c > maxstreaks[wcaid]:
                maxstreaks[wcaid] = c
                maxstart[wcaid] = startcomps[wcaid]
                maxend[wcaid] = "ongoing"
        else:
            c = currentstreak[wcaid]
            if c > maxstreaks[wcaid]:
                maxstreaks[wcaid] = c
                maxstart[wcaid] = startcomps[wcaid]
                maxend[wcaid] = currentcomp[wcaid]
                
    # Create the dataframe
    names = [personnames[wcaid] for wcaid in gotpr]
    maxstreak = [maxstreaks[wcaid] for wcaid in gotpr]
    maxstarts = [maxstart[wcaid] for wcaid in gotpr]
    maxends = [maxend[wcaid] for wcaid in gotpr]
    maxdays = [(compdates[maxends[i]] - compdates[maxstarts[i]]).days for i, _ in enumerate(maxends)]
    df = pd.DataFrame({"name":names, "maxstreak":maxstreak, "startcomp":maxstarts, "endcomp":maxends, 
    "days":maxdays})
    df = df.sort_values("maxstreak", ascending=False).reset_index(drop="index")
    df.to_csv(f"maxprstreak{country}.csv", index=False)


### Result

In [7]:
if __name__ == "__main__":
    # Longest PR Streak in Country
    main()

Pr streaks

In [8]:
a = pd.read_csv('maxprstreakItaly.csv') #change the name if you change the country to 'maxprstreak[country].csv'
a.index += 1
a.head(20)

,name,maxstreak,startcomp,endcomp,days
1,Simone Cantarelli,43,SchioOpen2012,MilanSaturday2018,2204
2,Matteo Dummar,43,SunmarkeDubaiOpenII2017,ongoing,1709
3,Marco Rota,38,ObeiObeiOpen2009,MantuaTinyOpen2016,2252
4,Mariano D'Imperio,33,FrenchOpen2009,RomeSlamOpen2016,2556
5,Marco Belotti,31,HalloweenOpen2010,ongoing,4256
6,Luca Brasini,31,FioranoOpen2017,ongoing,1925
7,Claudio Bentivoglio,29,LegnanoOpen2014,ongoing,2955
8,Carolina Guidetti,29,AlmostRomeOpen2017,OneLookinginCavarzere2022,1840
9,Mattia Furlan,28,LjubljanaOpen2013,ongoing,3193
10,Vittorio Maria Perucatti,27,MonterotondoOpen2015,PaladiamanteOpen2022,2289


Ongoing PR streaks

In [9]:
a[a['endcomp']=='ongoing'].reset_index(drop=True).head(20)

,name,maxstreak,startcomp,endcomp,days
0,Matteo Dummar,43,SunmarkeDubaiOpenII2017,ongoing,1709
1,Marco Belotti,31,HalloweenOpen2010,ongoing,4256
2,Luca Brasini,31,FioranoOpen2017,ongoing,1925
3,Claudio Bentivoglio,29,LegnanoOpen2014,ongoing,2955
4,Mattia Furlan,28,LjubljanaOpen2013,ongoing,3193
5,Marco Minici,26,TrentinOpen2011,ongoing,3879
6,Roberto Bentivoglio,21,MonticelloConteOtto2014,ongoing,3011
7,Nicola Barbaro,19,RomaWinterOpen2011,ongoing,3850
8,Ciro Vignotto,19,PoliMiItalianOpen2014,ongoing,2752
9,Riccardo Fasan,18,FioranoOpen2017,ongoing,1925
